### Logistic Regression: Week 4
#### Step 1: Load and Check the data

In [44]:
import pandas as pd

df = pd.read_csv("datasets/train.csv")

print(df.head())
print(df.info())
print(df.describe())
print(df["Embarked"].head(20))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

#### Step 2: fill Null values

In [45]:
# Fill age with median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Embarked with Mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop Cabin
df = df.drop(columns=["Cabin"])

#### Step 3: Feature Engineering

Phase 3 needs Titles and other features extracted before dropping uneeded columns

In [ ]:
# 1️ Extract Title FIRST
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Group rare titles
rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
df["Title"] = df["Title"].replace(rare_titles, "Other")

# 2️ Create FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# 3️ Create IsAlone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# 4️⃣ NOW drop unnecessary columns
df = df.drop(columns=["PassengerId", "Name", "Ticket"])

#### Step 4: One-Hot Encode

In [47]:
df = pd.get_dummies(df, drop_first=True)

#### Step 5: Train-Test Split

In [48]:
from sklearn.model_selection import train_test_split

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train,  X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#### Step 6: Scale (Important for Logisitic Regression)

In [49]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### Step 7: Logitistic Regression

In [50]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

#### Step 8: Evaluate

In [51]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8100558659217877
[[90 15]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



### Part 2
#### Step 1: Tune Regulatization with `LogisticRegressionCV`
Before we were doing default regularization, now we will let the model choose the best C automatically.

What this does:
* C is Inverse regularization strength:
    * Small C → Stronger regularization
    * Large C → Weaker regularization

What is C?:
* `Cs=[0.001, 0.01, 0.1, 1, 10, 100]`
* In Logistic regerssion `C = inverse of regularization strength`
    * C=1/λ​
    * Where:
        * `λ​ = regularization strength`
        * `C = how much freedom the model has`

What is CV?:
* CV is Cross-validation
* `cv = 5` splits my training data into 5 parts:
    * Train on 4 folds
    * Validate on 1 fold
    * Repeat 5 times
    * Average performance
    * Pick best C
* So instead of guessing regularization strength, the model chooses it statistically.

What is `penalty = "l2"`
* This defines the type of regulariation
    * L2 penalty adds `λ ∑ w²`
* What does L2 do?
    * Shrinks coefficients toward zero
    * Keeps all features
    * Reduces variance
    * Smooths model
* It does NOT eliminate feautres

Alternatively: "l1":
* This adds : `λ ∑ ∣w∣`
* L1:
    * Can shrink some coefficients exactly to 0
    performs feature selection
    Produces sparse models

What is `solver = "lbfgs"`:
* The solver is the optimization algorithim
* Logistic regression is solved via numerical optimization.
* `lbgs`:
    * Stands for: `Limited-Memory Boryden-Fletcher-Goldfarb-Shano`
    * Efficient for medium-sized datasets
    * Supports L2
    * Default for sklearn

In [52]:
from sklearn.linear_model import LogisticRegressionCV

log_cv = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l2",
    solver="lbfgs",
    max_iter=5000,
    random_state=42
)

log_cv.fit(X_train_scaled, y_train)

y_pred_cv = log_cv.predict(X_test)
log_cv_acc = accuracy_score(y_test, y_pred_cv)
best_c = log_cv.C_[0]

print("Accuracy:", accuracy_score(y_test, y_pred_cv))
print("Best C:", log_cv.C_[0])
print(confusion_matrix(y_test, y_pred_cv))
print(classification_report(y_test, y_pred_cv))

Accuracy: 0.664804469273743
Best C: 0.046415888336127774
[[100   5]
 [ 55  19]]
              precision    recall  f1-score   support

           0       0.65      0.95      0.77       105
           1       0.79      0.26      0.39        74

    accuracy                           0.66       179
   macro avg       0.72      0.60      0.58       179
weighted avg       0.71      0.66      0.61       179



c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-le

#### Step 2: Compare Random Forest
Random forest is a non-linear model.

Differences:
* Logistic Regression has Linear decision boundaries
* Random forest is nonlinear, captures interactions automatically, and handles Mixed features naturally


In [53]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

RF Accuracy: 0.8100558659217877
[[90 15]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



#### Step 3: Feature Importance (Random forest)

In [54]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

Fare          0.272564
Sex_male      0.270008
Age           0.253504
Pclass        0.083473
SibSp         0.050061
Parch         0.037864
Embarked_S    0.022906
Embarked_Q    0.009621
dtype: float64


#### Step 4: Interpret Logistic Regression Coefficients

In [55]:
coef_df = pd.Series(
    log_cv.coef_[0],
    index=X_train.columns
).sort_values()

print(coef_df)

Sex_male     -1.039022
Pclass       -0.583729
Age          -0.266995
SibSp        -0.243610
Embarked_S   -0.164094
Parch        -0.053115
Embarked_Q   -0.032848
Fare          0.160725
dtype: float64


In [56]:
# Final Comparison Output
print("=== Model Comparison ===")
print(f"LogisticRegressionCV Accuracy: {log_cv_acc:.4f}")
print(f"Best C (Regularization Strength): {best_c}")
print(f"RandomForest Accuracy: {rf_acc:.4f}")

=== Model Comparison ===
LogisticRegressionCV Accuracy: 0.6648
Best C (Regularization Strength): 0.046415888336127774
RandomForest Accuracy: 0.8101


#### Insight:
* Linear models are great for interpretability, but require careful feature engineering (e.g., Titles, FamilySize, IsAlone).
* **RandomForests** handle complex relationships automatically
* Key Lessons in ML: Model choice matters and sometimes adding features is as important as tuning hyperparameters.

### Phase 3
#### Step 1: Create New Features

In [ ]:
# --- Feature Engineering on full dataframe ---

# 1️⃣ Extract Title
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

rare_titles = ['Lady','Countess','Capt','Col','Don','Dr',
               'Major','Rev','Sir','Jonkheer','Dona']

df["Title"] = df["Title"].replace(rare_titles, "Other")

# 2️⃣ Family Size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# 3️⃣ Is Alone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# 4️⃣ NOW drop unused raw columns
df = df.drop(columns=["PassengerId", "Name", "Ticket"])

c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
c:\Users\asume\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-le

ValueError: could not convert string to float: 'male'

#### Step 2: Encode Categorical Features
* One-hot encode Sex, Embarked, Title
* Drop the first category to avoid dummy variable trap

In [ ]:


# one-hot encode
categorical_cols = ["Sex", "Embarked", "Title"]
X_train3 = pd.get_dummies(X_train3, columns=categorical_cols, drop_first=True)
X_test3 = pd.get_dummies(X_test3, columns=categorical_cols, drop_first=True)

#### Step 3: Train LogisticRegressionCV Again

In [ ]:
# align columns (important)
X_train3, X_test3 = X_train3.align(X_test3, join="left", axis=1, fill_value=0)

# train
log_cv3 = LogisticRegressionCV(
    Cs=10, cv=5, penalty="l2", solver="lbfgs", max_iter=5000, random_state=42
)
log_cv3.fit(X_train3, y_train3)

y_pred3 = log_cv3.predict(X_test3)

print("Accuracy:", accuracy_score(y_test3, y_pred3))
print("Best C:", log_cv3.C_[0])
print(confusion_matrix(y_test3, y_pred3))
print(classification_report(y_test3, y_pred3))
